In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:90%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

```
1. csv 데이터 불러오기 
2. 임베딩
3. 사용자 입력받기 
4. 유사도 계산
5. csv 파일로 저장

```

In [2]:
from sentence_transformers import SentenceTransformer,util

model = SentenceTransformer('all-MiniLM-L6-v2')
print("모델 로드 성공!")

C:\Users\Admin\anaconda3\envs\mia\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



모델 로드 성공!


In [ ]:
# 1. 모델 로드

In [31]:
import pandas as pd
# model = SentenceTransformer('all-MiniLM-L6-v2') # 빠르고 가볍지만 문맥정밀도는 떨어짐
model = SentenceTransformer('paraphrase-mpnet-base-v2')

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


CPU times: total: 1.88 s
Wall time: 30.6 s


In [ ]:
# 2. csv 파일 불러오기

In [4]:
df = pd.read_csv('data/dating_circular_talkset_10k_cp949 (1).csv', encoding='cp949')
df.head(2)

,질문자성별,질문,답변자성별,답변,리액션_추가질문
0,여성,최근에 재밌게 본 드라마 (요즘) 있으세요?,남성,모바일 게임 '쿠키런'에 빠졌어요.,게임에서 가장 좋아하는 캐릭터는 뭐예요?
1,여성,최근에 재밌게 본 드라마 (요즘) 있으세요?,남성,요즘 다이어트 도시락에 빠져 있어요.,도시락 직접 만드시나요? 추천 레시피 있으세요?


In [5]:
%%time
# 3. 입력_내용 임베딩
input_texts=df['질문'].fillna("").tolist()
# input_texts
input_embeddings = model.encode(input_texts, convert_to_tensor=True)
input_embeddings

C:\Users\Admin\anaconda3\envs\mia\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


CPU times: total: 1h 55min 50s
Wall time: 32min 3s


tensor([[-0.0289,  0.0327,  0.0269,  ...,  0.0900, -0.0241, -0.0495],
        [-0.0289,  0.0327,  0.0269,  ...,  0.0900, -0.0241, -0.0495],
        [-0.0289,  0.0327,  0.0269,  ...,  0.0900, -0.0241, -0.0495],
        ...,
        [-0.0233,  0.0700,  0.0742,  ...,  0.0264,  0.0062,  0.0139],
        [-0.0233,  0.0700,  0.0742,  ...,  0.0264,  0.0062,  0.0139],
        [-0.0233,  0.0700,  0.0742,  ...,  0.0264,  0.0062,  0.0139]])

In [32]:
%%time
# 4. 사용자 입력 받기

user_input = "최근에 무슨 드라마 보셨어요?"
user_embedding= model.encode(user_input, convert_to_tensor=True)

CPU times: total: 500 ms
Wall time: 142 ms


In [33]:
%%time
cos_scores = util.cos_sim(user_embedding, input_embeddings)[0]
top_idx = cos_scores.argmax()

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x768 and 384x200000)

In [34]:
%time
top_idx_int = top_idx.item()
print("🎯 추천 질문:", df.loc[top_idx_int, '리액션_추가질문'])
print("📌 원래 입력 예시:", df.loc[top_idx_int, '질문'])
print("💬 챗봇 응답:", df.loc[top_idx_int, '답변'])
print("📈 유사도 점수:", cos_scores[top_idx_int].item())

CPU times: total: 0 ns
Wall time: 0 ns
🎯 추천 질문: 단골카페 분위기 좋죠! 추천 음료 있으세요?
📌 원래 입력 예시: 티빙/웨이브/디즈니+ 등 OTT에서 보는 콘텐츠 (즐겨보는) 있으세요?
💬 챗봇 응답: 자주 가는 동네 카페가 있어요.
📈 유사도 점수: 0.78412264585495


In [27]:
%%time
# 4. 사용자 입력 받기

user_input = "OTT는 어떤거 보시나요 ?"
user_embedding= model.encode(user_input, convert_to_tensor=True)

CPU times: total: 0 ns
Wall time: 23.3 ms


In [28]:
%%time
cos_scores = util.cos_sim(user_embedding, input_embeddings)[0]
top_idx = cos_scores.argmax()

CPU times: total: 312 ms
Wall time: 78.7 ms


In [29]:
%time
top_idx_int = top_idx.item()
print("🎯 추천 질문:", df.loc[top_idx_int, '리액션_추가질문'])
print("📌 원래 입력 예시:", df.loc[top_idx_int, '질문'])
print("💬 챗봇 응답:", df.loc[top_idx_int, '답변'])
print("📈 유사도 점수:", cos_scores[top_idx_int].item())

CPU times: total: 0 ns
Wall time: 0 ns
🎯 추천 질문: 단골카페 분위기 좋죠! 추천 음료 있으세요?
📌 원래 입력 예시: 티빙/웨이브/디즈니+ 등 OTT에서 보는 콘텐츠 (즐겨보는) 있으세요?
💬 챗봇 응답: 자주 가는 동네 카페가 있어요.
📈 유사도 점수: 0.78412264585495
